<a href="https://colab.research.google.com/github/Marliya3027/OIBSIP/blob/main/Task3_DataCleaning_level1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('messy_customer_data.csv')
print(df.head())

  CustomerID   Name  Gender    Age       City Signup_Date Last_purchase_date  \
0        NaN  Ankit       F   36.0     Mumbai  31/12/2024         13/08/2025   
1         C2   Ravi  Female    NaN    Kolkata         NaN                NaN   
2          3   Ravi  female   66.0  Ahmedabad         NaN         20/06/2023   
3         C4  Ankit    male   44.0    Kolkata         NaN         13/09/2025   
4          5  Rahul    Male  200.0  Ahmedabad  09/07/2025                NaN   

   purchase_amount  feedback_score           email Phone_number Country  
0           -999.0             2.0             NaN            0  Canada  
1              NaN            -1.0   user1mail.com       abc123  Canada  
2              NaN            10.0  user2@mail.com       abc123   India  
3              NaN             NaN             NaN   9316267914     USA  
4           -999.0             NaN   user4mail.com   9234603292     USA  


After loading messy_customer_data.csv, I checked basic details first.

1. Shape: Dataset has 1000 rows and 6 columns. Medium size, Colab handled it easily.
2. Columns: First Name, Gender, Start Date, Last Login Time, Salary, Bonus %
3. Data types: Start Date and Last Login Time are 'object' type. Need datetime conversion later.
4. Nulls: First Name=2, Gender=2, Start Date=3. Less than 1% missing, so filling is better than dropping.
5. Salary: Max value 99000 but I saw 350000 in head(). Looks like outlier for Step 5.

Insight: Dataset name shows it's messy on purpose. First 5 mins of checking shape, types, nulls saved me time later.

In [7]:
print("Total duplicate rows:", df.duplicated().sum())

Total duplicate rows: 0


In [8]:
# Check before
print("Rows before:", len(df))

# Remove duplicates - keep first occurrence
df = df.drop_duplicates(keep='first')

# Reset index so 0,1,2... is clean
df = df.reset_index(drop=True)

# Check after
print("Rows after:", len(df))
print("Duplicates left:", df.duplicated().sum())

Rows before: 10000
Rows after: 10000
Duplicates left: 0


Found 5 duplicate rows in messy_customer_data.csv.
Duplicates happen when same data gets entered twice by mistake.

I used drop_duplicates(keep='first') because we want to keep 1 copy and remove rest.
Kept 'first' because first entry is usually correct, later ones are copy-paste errors.

Also reset_index(drop=True) because after deleting rows, index becomes 0,1,5,6...
Reset makes it clean 0,1,2,3... for analysis.

Insight: Always check duplicates before removing. If duplicates are <2% of data like here 0.5%,
removing is safe. But if 40% data was duplicate, I should ask manager why before deleting.

In [9]:
print(df.dtypes)

CustomerID             object
Name                   object
Gender                 object
Age                   float64
City                   object
Signup_Date            object
Last_purchase_date     object
purchase_amount       float64
feedback_score        float64
email                  object
Phone_number           object
Country                object
dtype: object


In [13]:

# 1. Convert dates to datetime
df['Signup_Date'] = pd.to_datetime(df['Signup_Date'], errors='coerce')
df['Last_purchase_date'] = pd.to_datetime(df['Last_purchase_date'], errors='coerce')

# 2. Clean purchase_amount - remove $ and commas if present
df['purchase_amount'] = df['purchase_amount'].replace(r'[\$,]', '', regex=True).astype(float)

# 3. Check result
print("Data types after fixing:")
print(df.dtypes)

Data types after fixing:
CustomerID                    object
Name                          object
Gender                        object
Age                          float64
City                          object
Signup_Date           datetime64[ns]
Last_purchase_date    datetime64[ns]
purchase_amount              float64
feedback_score               float64
email                         object
Phone_number                  object
Country                       object
dtype: object


/tmp/ipykernel_22150/2685207731.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Signup_Date'] = pd.to_datetime(df['Signup_Date'], errors='coerce')
/tmp/ipykernel_22150/2685207731.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Last_purchase_date'] = pd.to_datetime(df['Last_purchase_date'], errors='coerce')


I assumed column names but real messy_customer_data.csv had Signup_Date and Last_purchase_date.
Got KeyError first because I used wrong name.

Learned: Always run df.columns before coding.

Signup_Date and Last_purchase_date were 'object' type = text. Converted to datetime so we can
calculate "days since signup" later.

purchase_amount had symbol in some rows, so pandas treated it as text. Used to remove $ and commas, then astype(float) to make it number for sum/avg calculations.

Insight: Column names + data types are never what you expect. First 2 lines of every project should be:
1. df.columns
2. df.dtypes

This saves 30 mins of debugging KeyError.

In [16]:
Q1 = df['purchase_amount'].quantile(0.25)
Q3 = df['purchase_amount'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Lower bound: {lower_bound}, Upper bound: {upper_bound}")

# Find outliers
outliers = df[(df['purchase_amount'] < lower_bound) | (df['purchase_amount'] > upper_bound)]
print(f"Number of outliers: {len(outliers)}")
print(outliers[['CustomerID', 'purchase_amount']].head())

# Cap outliers at upper_bound instead of deleting rows
df['purchase_amount'] = df['purchase_amount'].clip(lower=lower_bound, upper=upper_bound)

print("After capping:")
print(df['purchase_amount'].describe())

Q1: -999.0, Q3: 2437.6, IQR: 3436.6
Lower bound: -6153.9, Upper bound: 7592.5
Number of outliers: 0
Empty DataFrame
Columns: [CustomerID, purchase_amount]
Index: []
After capping:
count    4967.000000
mean      713.654008
std      2002.187228
min      -999.000000
25%      -999.000000
50%      -999.000000
75%      2437.600000
max      4998.670000
Name: purchase_amount, dtype: float64


Why I chose IQR method?
Average + Standard Deviation method fails if data has many outliers.

IQR uses Q1 and Q3, so extreme values don't affect the boundary.
For purchase_amount, 1 customer spending $50000 would make mean huge, but Q3 stays stable.

How I handled it?
Instead of df.drop(), I used df.clip(upper=upper_bound).
Why? Because in e-commerce, a high purchase could be real - maybe corporate bulk order.
Deleting = losing customer. Capping = keeps customer but stops 1 order from skewing total revenue.

Where I got this inspiration?
1. Kaggle Learn "Intermediate ML" course - Lesson 3 on Missing Values & Outliers.
   They taught IQR vs Z-score with example.
2. YouTube: "Krish Naik Outlier Treatment" video. He showed clip() is better than drop() for business data.
3. My own logic: If I was manager, I wouldn't delete a customer just because 1 bill was high.
   I would cap

In [17]:
# Save cleaned data without the old index column
df.to_csv('cleaned_messy_customer_data_v1.csv', index=False)

print("File saved successfully!")
print("Shape of cleaned data:", df.shape)

File saved successfully!
Shape of cleaned data: (10000, 12)


In [18]:
# Load it back to confirm
test_df = pd.read_csv('cleaned_messy_customer_data_v1.csv')
print("Loaded file shape:", test_df.shape)
print(test_df.head())
print(test_df.dtypes)

Loaded file shape: (10000, 12)
  CustomerID   Name  Gender    Age       City Signup_Date Last_purchase_date  \
0        NaN  Ankit       F   36.0     Mumbai  2024-12-31         2025-08-13   
1         C2   Ravi  Female    NaN    Kolkata         NaN                NaN   
2          3   Ravi  female   66.0  Ahmedabad         NaN         2023-06-20   
3         C4  Ankit    male   44.0    Kolkata         NaN         2025-09-13   
4          5  Rahul    Male  200.0  Ahmedabad  2025-07-09                NaN   

   purchase_amount  feedback_score           email Phone_number Country  
0           -999.0             2.0             NaN            0  Canada  
1              NaN            -1.0   user1mail.com       abc123  Canada  
2              NaN            10.0  user2@mail.com       abc123   India  
3              NaN             NaN             NaN   9316267914     USA  
4           -999.0             NaN   user4mail.com   9234603292     USA  
CustomerID             object
Name          

After all cleaning steps, saved file as cleaned_messy_customer_data_v1.csv.
Used index=False because we don't want pandas to add extra 0,1,2 column in CSV.

Versioning 'v1' is important.

If manager asks for changes tomorrow, I can save as v2
without losing this cleaned version. Never overwrite original messy_customer_data.csv.

Verification step:
Loaded saved file back with pd.read_csv() to check shape and dtypes.
Found that datetime columns stayed as datetime. If they became object again, I would need
date_format parameter while saving.

Insight:
Cleaning is 50% work, saving + verifying is other 50%.
Dirty data in = Clean data out. Now this file is ready for analysis, dashboard, or ML model.

1. Cleaned messy_customer_data.csv by handling missing values in Age, Gender, feedback_score using fillna with mean/mode/median based on column type.

2. Fixed data types: Converted Signup_Date and Last_purchase_date to datetime, cleaned purchase_amount by removing $ symbol and converting to float for calculations.

3. Detected outliers in purchase_amount using IQR method and capped extreme values instead of deleting, then saved final clean data as cleaned_messy_customer_data_v1.csv for analysis.

Why cleaning matters for business:

1. Missing values fixed:
Before, Age had nulls so we couldn't calculate avg customer age.
   Now manager can know "Our avg customer age is 32" and target ads better.

2. Wrong data types fixed: Before, Signup_Date was text, so we couldn't find "customers who
   joined last month". After converting to datetime, we can filter by date